In [ ]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
def data_to_model(data, sequence_length=120, step_size=25):
    data = torch.tensor(data, dtype=torch.float32) if not isinstance(data, torch.Tensor) else data
    seqs = [data[i:i+sequence_length] for i in range(0, len(data)-sequence_length+1, step_size)]
    return torch.stack(seqs)


In [ ]:
DATA_ROOT = r'...'

def get_sensor_data(mode, phase, label, shuffle=True, ratio=1, sequence_length=80, step_size=4):
    all_data, all_labels = [], []
    for m, p in zip(mode, phase):
        raw = pd.read_csv(os.path.join(DATA_ROOT, m, f'all_sycn_raw_data_mode{p}.csv')).to_numpy()
        err = pd.read_csv(os.path.join(DATA_ROOT, m, f'all_sycn_err_data_mode{p}.csv')).to_numpy()
        rs, es = np.split(raw, 3, axis=1), np.split(err, 3, axis=1)
        r_ = np.hstack((rs[1], rs[0], rs[2]))
        e_ = np.hstack((es[1], es[0], es[2]))
        r2 = [r_[:,0:3], r_[:,3:6], r_[:,6:9]]
        e2 = [e_[:,0:3], e_[:,3:6], e_[:,6:9]]
        raw_err = np.hstack((r2[0],e2[0],r2[1],e2[1],r2[2],e2[2]))
        seqs = data_to_model(data=raw_err, sequence_length=sequence_length, step_size=step_size)
        labs = torch.full((seqs.size(0), 1), label, dtype=torch.long)
        all_data.append(seqs); all_labels.append(labs)
    raw_err_2_model = torch.cat(all_data, dim=0)
    raw_err_2_model_labels = torch.cat(all_labels, dim=0)
    if shuffle:
        torch.manual_seed(42)
        indices = torch.randperm(raw_err_2_model.size(0))
        raw_err_2_model = raw_err_2_model[indices]
        raw_err_2_model_labels = raw_err_2_model_labels[indices]
    raw_err_2_model = raw_err_2_model[:int(raw_err_2_model.size(0) * ratio)]
    raw_err_2_model_labels = raw_err_2_model_labels[:int(raw_err_2_model_labels.size(0) * ratio)]
    return raw_err_2_model, raw_err_2_model_labels

normal_2_model, normal_lables = get_sensor_data(mode=['1-hover-normal-r2vo', '2-forward-normal-r2vo', '1-hover-normal-r2vo'], phase=['2_6', '2_5', '2_3'], label=1)
af3_2_model, af3_labels = get_sensor_data(mode=['3-acc-x-axis-flag3-3000-normal-r2vo'], phase=['2_9'], label=2)
mf3_2_model, mf3_labels = get_sensor_data(mode=['6-motor03-flag3-200-normal-r2vo'], phase=['2_9'], label=3)
mf4_2_model, mf4_labels = get_sensor_data(mode=['5-motor03-flag4-085-normal-r2vo'], phase=['2_9'], label=4)

print(f'normal:{normal_2_model.shape} C2:{af3_2_model.shape} C3:{mf3_2_model.shape} C4:{mf4_2_model.shape}')

data_2_model = torch.cat((normal_2_model, af3_2_model, mf3_2_model, mf4_2_model), dim=0)
labels_2_model = torch.cat((normal_lables, af3_labels, mf3_labels, mf4_labels), dim=0).squeeze(1)

num_classes = 4
one_hot = torch.zeros(labels_2_model.size(0), num_classes, dtype=torch.float)
for i in range(labels_2_model.size(0)):
    one_hot[i][labels_2_model[i] - 1] = 1
label_2_model = one_hot
print(f'data:{data_2_model.shape} labels:{label_2_model.shape}')


In [ ]:
class CNN_Transformer_Classifier(nn.Module):
    """CNN-Transformer Hybrid Network (Paper [3])"""
    def __init__(self, in_channels=18, seq_len=80, num_class=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        encoder_layer = nn.TransformerEncoderLayer(d_model=in_channels, nhead=6,
                                                    dim_feedforward=72, batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.pos_emb = nn.Parameter(torch.randn(1, seq_len, in_channels)*0.02)
        self.trans_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(128 + in_channels, 128)
        self.fc2 = nn.Linear(128, num_class)
    def forward(self, x):
        c = self.cnn(x.permute(0, 2, 1)).squeeze(-1)
        t = x + self.pos_emb[:, :x.size(1), :]
        t = self.transformer(t)
        t = self.trans_pool(t.permute(0, 2, 1)).squeeze(-1)
        h = torch.cat([c, t], dim=1)
        h = F.relu(self.fc1(h))
        return self.fc2(h)


In [ ]:
def train(model, train_data, test_data, batch_size, optimizer, criterion, device):
    model.train()
    x_train, y_train = train_data
    x_test, y_test = test_data
    total_loss = 0
    indices = torch.randperm(len(x_train))
    for start in range(0, len(x_train), batch_size):
        idx = indices[start:start+batch_size]
        xb = x_train[idx].to(device)
        yb = y_train[idx].to(device)
        out = model(xb)
        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)
    train_loss = total_loss / len(x_train)
    # Test loss
    model.eval()
    with torch.no_grad():
        test_out = []
        for start in range(0, len(x_test), batch_size):
            xb = x_test[start:start+batch_size].to(device)
            test_out.append(model(xb))
        test_out = torch.cat(test_out, dim=0)
        test_loss = criterion(test_out, y_test.to(device)).item()
    return train_loss, test_loss


In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_data = data_2_model
train_label = label_2_model
print('Train data shape:', train_data.shape)

train_ratio = 0.7
x_train, x_test = train_test_split(train_data, train_size=train_ratio, shuffle=True, random_state=seed)
y_train, y_test = train_test_split(train_label, train_size=train_ratio, shuffle=True, random_state=seed)

train_2_model = [x_train, y_train]
test_2_model = [x_test, y_test]
print(f'x_train: {x_train.shape} x_test: {x_test.shape}')
print(f'y_train: {y_train.shape} y_test: {y_test.shape}')


In [ ]:
ReCall = False
if ReCall:
    folder_name = 'FD_CNN_Transformer'
    model_name = 'FD_CNN_Transformer.pth'
    model_path = os.path.join(os.getcwd(), 'info', folder_name, model_name)
    model = torch.load(model_path)
    model.to(device)
else:
    in_channels = x_train.shape[2]
    model = CNN_Transformer_Classifier(in_channels=in_channels, seq_len=80, num_class=4)
    model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epoch_train_loss = np.array([])
epoch_test_loss = np.array([])


In [ ]:
num_epochs = 30
batch_size = 64

for epoch in range(num_epochs):
    train_loss, test_loss = train(model, train_2_model, test_2_model, batch_size, optimizer, criterion, device)
    epoch_train_loss = np.append(epoch_train_loss, train_loss)
    epoch_test_loss = np.append(epoch_test_loss, test_loss)
    print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

In [ ]:
plt.plot(epoch_train_loss, label='Train Loss')
plt.plot(epoch_test_loss, label='Test Loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN_Transformer Loss Curve')
plt.show()


In [ ]:
def calculate_accuracy(data, label):
    model.eval()
    acc = 0
    ori_lab = label
    out_lab = np.array([0,0,0,0])
    with torch.no_grad():
        for i in range(len(data)):
            orign = data[i].unsqueeze(0).to(device)
            lab = label[i].numpy()
            out = model(orign).cpu().numpy().flatten()
            out_lab = np.vstack((out_lab, out))
            predicted_classes = np.argmax(out)
            if lab[predicted_classes] == 1:
                acc += 1
    accuracy = acc / len(data)
    out_lab = out_lab[1:]
    return accuracy, ori_lab, out_lab

train_accuracy, ori_train_lab, out_train_lab = calculate_accuracy(x_train, y_train)
test_accuracy, ori_test_lab, out_test_lab = calculate_accuracy(x_test, y_test)
print(f'train_accuracy: {train_accuracy}')
print(f'test_accuracy: {test_accuracy}')


In [ ]:
from sklearn.metrics import confusion_matrix

train_cm = confusion_matrix(np.argmax(ori_train_lab, axis=1), np.argmax(out_train_lab, axis=1))
TP = np.diag(train_cm)
FP = np.sum(train_cm, axis=0) - TP
FN = np.sum(train_cm, axis=1) - TP
TN = np.sum(train_cm) - (TP + FP + FN)
print('Train Dataset:')
precision = np.mean(TP / (TP + FP))
recall = np.mean(TP / (TP + FN))
f1_score = 2 * (precision * recall) / (precision + recall)
accuracy = np.sum(TP) / np.sum(train_cm)
print('Precision: {:.4f}  Recall: {:.4f}  F1 Score: {:.4f}  Accuracy: {:.4f}'.format(precision, recall, f1_score, accuracy))

test_cm = confusion_matrix(np.argmax(ori_test_lab, axis=1), np.argmax(out_test_lab, axis=1))
print('\nTest Dataset:')
TP = np.diag(test_cm)
FP = np.sum(test_cm, axis=0) - TP
FN = np.sum(test_cm, axis=1) - TP
TN = np.sum(test_cm) - (TP + FP + FN)
precision = np.mean(TP / (TP + FP))
recall = np.mean(TP / (TP + FN))
f1_score = 2 * (precision * recall) / (precision + recall)
accuracy = np.sum(TP) / np.sum(test_cm)
print('Precision: {:.4f}  Recall: {:.4f}  F1 Score: {:.4f}  Accuracy: {:.4f}'.format(precision, recall, f1_score, accuracy))


In [ ]:
import shutil

ReSaved = True
mode_name = 'FD_CNN_Transformer.pth'
folder_name = mode_name.split('.')[0]
folder_path = os.path.join(os.getcwd(), 'info', folder_name)
path = os.path.join(folder_path, mode_name)

if os.path.exists(folder_path):
    if ReSaved:
        shutil.rmtree(folder_path)
        os.makedirs(folder_path, exist_ok=True)
        print(f'Model Re-Saved to {path}')
    else:
        print(f'Model Co-Saved to {path}')
else:
    os.makedirs(folder_path, exist_ok=True)
    print(f'Model Fi-Saved to {path}')

torch.save(model, path)

# Save loss
train_loss = epoch_train_loss
test_loss = epoch_test_loss
loss_combined = np.column_stack((train_loss, test_loss))
df1 = pd.DataFrame(loss_combined, columns=['train_loss', 'test_loss'])
file_name = os.path.join(folder_path, 'loss.csv')
df1.to_csv(file_name, index=False)
print(f'Saved loss to {file_name}')

# Save accuracy
accuracy_combined = np.column_stack((train_accuracy, test_accuracy))
df2 = pd.DataFrame(accuracy_combined, columns=['train_accuracy', 'test_accuracy'])
file_name = os.path.join(folder_path, 'accuracy.csv')
df2.to_csv(file_name, index=False)
print(f'Saved accuracy to {file_name}')
